# E08 · 02 Semantic Scholar 元数据与引用关系

**这个 notebook 做的事**：给一份人工挑好的论文清单，每篇解析出 S2 的 `paperId`，然后
**一篇一次请求**，把 S2 能给的元数据、作者、参考文献列表、施引文献列表整个拿回来落盘。

构建数据集期间它就是这样用的：把要准备的论文填进 `PAPERS`，跑一遍，`_scratch/s2/` 下
每篇一个原样 JSON。后期系统构建时，`resolve_entry()` + `fetch_paper()` 这两个函数就是
operator 里"元数据获取"那一环要融进去的东西，接口不用变。

**不做的事**：不做检索发现（论文是你选的）、不做跨篇整合、不做质量评估。这些属于别处。

**口径声明**

- 拿到的是 **S2 的记录**，不是论文本身的事实。`citationCount` 等计数是**请求时刻的
  快照**，`fetched_at` 一并存下，引用数字时必须带上它。
- S2 把预印本和正式版**合并成一条**，但字段可能来自不同版本：实测 DLinear 那条
  `venue` 是 AAAI、`year` 是 2022（arXiv 年）、`DOI` 是 `10.48550/arXiv.*`（AAAI 官方
  DOI 反而查不到）。所以 **S2 的 `venue` / `year` 不能直接当作发表信息**落库。
- 入口优先级 **arXiv ≥ DOI > 标题**。DOI 入口会因为出版方 DOI 不在 S2 而 404。
- 标题入口要**验收**：返回标题与手里的标题归一化后必须相等。`search/match` 对编造标题、
  缩写、OCR 噪声都返回 404（保守），唯一的静默错误是 query 恰好是另一篇标题的前缀
  （实测 `Learned Index` → `G-Learned Index: ...`），归一化相等这一条就能挡住。
- **解析只做一次**：拿到 `paperId` 就回填进清单，之后一律按 id 走，抓取不再依赖 S2 的
  匹配行为。
- 一次请求的两个上限：嵌套的 `references` / `citations` **各最多 1000 条且不能分页**
  （超了要改用 `/paper/{id}/citations?offset=`）；引用的**边属性**（`contexts` /
  `intents` / `isInfluential`）在单篇端点上不可取（400），要它们得另外打那两个端点。
- key 取自 `experiments/e08/.env` 的 `S2K=`（或同名环境变量）。没有则退匿名池，限额更紧；
  走的哪条通道记在产物的 `key_origin` 里。带 key 也会随机 429，客户端自己退避重试。
- 探索面产物，**不是** `data/processed/` 的流水线工件。

In [ ]:
import json
import os
import random
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "AGENTS.md").exists():          # notebook 可能从任意目录启动
    REPO = REPO.parent

E08 = REPO / "experiments/e08"
OUT_DIR = Path.cwd() / "_scratch/s2"              # 探索产物，不进 data/processed

S2_BASE = "https://api.semanticscholar.org/graph/v1"
ENV_KEY = "S2K"


def resolve_api_key() -> tuple[str | None, str]:
    """返回 (key, 来源)。找不到返回 (None, "匿名") —— 匿名也能用，只是限额更紧。"""
    if os.environ.get(ENV_KEY):
        return os.environ[ENV_KEY], f"环境变量 {ENV_KEY}"
    for path in (Path.cwd() / ".env", E08 / ".env"):
        if not path.is_file():
            continue
        for raw in path.read_text(encoding="utf-8").splitlines():
            key, sep, value = raw.strip().partition("=")
            if sep and key.strip() == ENV_KEY and (value := value.strip().strip("\"'")):
                return value, str(path)
    return None, "匿名（无 key）"


S2_KEY, KEY_ORIGIN = resolve_api_key()
print("key 来源:", KEY_ORIGIN, f"({S2_KEY[:7]}…)" if S2_KEY else "")

## 客户端

最小间隔节流 + 对 429/5xx 指数退避。`404` 与 `400` 直接返回给调用方：前者是"没这篇"
（标题没匹配上也是 404），是结果不是故障；后者是字段名写错。

In [ ]:
MIN_INTERVAL = 1.2        # 官方限额 ~1 req/s（共享池），留一点余量
RETRY_CODES = {429, 500, 502, 503, 504}

_last_call = 0.0


def s2_call(path: str, params: dict | None = None, *, tries: int = 6) -> tuple[int, object]:
    """返回 (status, payload)；非 2xx 时 payload 是响应体前 300 字。"""
    global _last_call
    url = S2_BASE + path + (("?" + urllib.parse.urlencode(params)) if params else "")
    headers = {"x-api-key": S2_KEY} if S2_KEY else {}
    for attempt in range(tries):
        if (gap := MIN_INTERVAL - (time.monotonic() - _last_call)) > 0:
            time.sleep(gap)
        _last_call = time.monotonic()
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers),
                                        timeout=300) as resp:   # 高被引论文可能要 30s+
                return resp.status, json.loads(resp.read())
        except urllib.error.HTTPError as exc:
            if exc.code in RETRY_CODES and attempt < tries - 1:
                time.sleep(2 ** attempt + random.random())      # 1,2,4,8,16s(+抖动)
                continue
            return exc.code, exc.read().decode("utf-8", "replace")[:300]
    raise RuntimeError("unreachable")

## 清单

一行一篇，`key` 是 citekey（与 `references/refs.bib`、PDF、笔记共用同一个）。入口四选一，
按 `s2` → `arxiv` → `doi` → `title` 的顺序取**第一个填了的**：

| 入口 | 何时用 |
| --- | --- |
| `s2` | 已经解析过，回填进来的 `paperId`。有它就不再解析 |
| `arxiv` | 首选。arXiv 号最稳 |
| `doi` | 没有 arXiv 时。出版方 DOI 可能不在 S2 里（会 404，自动退到下一个入口） |
| `title` | 兜底。必须过标题验收 |

解析成功后把 `paperId` 抄回 `s2:` —— 这一步是手工的，就一次，之后这篇再也不走匹配。

In [ ]:
PAPERS = [
    {"key": "hnsw-tpami2020",   "arxiv": "1603.09320"},
    {"key": "digra-pacmmod2025", "doi": "10.1145/3725399"},
    {"key": "learned-index-sigmod2018",
     "title": "The Case for Learned Index Structures"},
    {"key": "alex-sigmod2020",  "arxiv": "1905.08898",
     "doi": "10.1145/3318464.3389711"},          # 两个都填也只会用到第一个成功的
]

norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())


def resolve_entry(entry: dict) -> dict:
    """清单条目 → {paper_id, resolved_by, s2_title, title_check, error}。"""
    out = {"key": entry["key"], "paper_id": None, "resolved_by": None,
           "s2_title": None, "title_check": None, "error": None}
    attempts = []

    if entry.get("s2"):
        return out | {"paper_id": entry["s2"], "resolved_by": "given"}
    if entry.get("arxiv"):
        attempts.append(("arxiv", f"/paper/arXiv:{entry['arxiv']}"))
    if entry.get("doi"):
        attempts.append(("doi", f"/paper/DOI:{urllib.parse.quote(entry['doi'], safe='/')}"))

    for via, path in attempts:
        status, payload = s2_call(path, {"fields": "paperId,title"})
        if status == 200:
            return out | {"paper_id": payload["paperId"], "resolved_by": via,
                          "s2_title": payload["title"]}
        if status != 404:
            return out | {"resolved_by": via, "error": f"{status}: {payload}"}

    if title := entry.get("title"):
        status, payload = s2_call("/paper/search/match", {"query": title, "fields": "paperId,title"})
        if status == 404:
            return out | {"resolved_by": "title", "error": "标题没匹配上"}
        if status != 200:
            return out | {"resolved_by": "title", "error": f"{status}: {payload}"}
        hit = payload["data"][0]
        # 验收：归一化后必须相等。挡住"query 是另一篇标题前缀"这唯一的静默错误形态
        check = "ok" if norm(hit["title"]) == norm(title) else "mismatch"
        return out | {"paper_id": hit["paperId"] if check == "ok" else None,
                      "resolved_by": "title", "s2_title": hit["title"], "title_check": check,
                      "error": None if check == "ok" else f"标题不符，返回的是：{hit['title']}"}

    return out | {"error": "没有可用入口"}


resolved = [resolve_entry(e) for e in PAPERS]
for r in resolved:
    print(f"  {r['key']:28} {r['resolved_by'] or '-':7} {r['paper_id'] or '—'}"
          + (f"  ⚠ {r['error']}" if r["error"] else ""))

pending = [r for r in resolved if not r["paper_id"]]
print(f"\n{len(resolved) - len(pending)}/{len(resolved)} 解析成功"
      + ("；下列要人工给 id：" + ", ".join(r["key"] for r in pending) if pending else ""))

## 一篇一次请求

`/paper/{id}?fields=...` 支持嵌套展开，所以主论文 + 作者 + 参考文献 + 施引文献
一个 GET 就够。字段分三组：`SELF` 只对主论文可用（`tldr` / `embedding` 展不到邻居上，
写了会整串 400），`AUTHOR` 是作者实体字段，`NEI` 是邻居论文的字段。

邻居字段挑多挑少直接决定响应体大小：实测 37 ref / 39 cite 的一篇，全字段 249 KiB、2.7s；
80 ref / 1314 cite 的一篇 3.5 MiB、33s（且施引被截到 1000）。嫌大就把 `NEI` 砍到
`paperId,externalIds,title,year,venue`。

In [ ]:
SELF = ["paperId", "corpusId", "externalIds", "url", "title", "abstract", "venue",
        "publicationVenue", "year", "publicationDate", "journal", "publicationTypes",
        "fieldsOfStudy", "s2FieldsOfStudy", "referenceCount", "citationCount",
        "influentialCitationCount", "isOpenAccess", "openAccessPdf", "citationStyles",
        "tldr", "embedding.specter_v2"]
AUTHOR = ["authorId", "externalIds", "url", "name", "affiliations", "homepage",
          "paperCount", "citationCount", "hIndex"]
NEI = ["paperId", "corpusId", "externalIds", "url", "title", "abstract", "venue",
       "publicationVenue", "year", "publicationDate", "journal", "publicationTypes",
       "fieldsOfStudy", "s2FieldsOfStudy", "referenceCount", "citationCount",
       "influentialCitationCount", "isOpenAccess", "openAccessPdf", "citationStyles",
       "authors"]

FIELDS = ",".join(SELF + [f"authors.{a}" for a in AUTHOR]
                  + [f"references.{x}" for x in NEI] + [f"citations.{x}" for x in NEI])
NESTED_CAP = 1000        # 嵌套列表的硬上限，且单篇端点没有 offset


def fetch_paper(paper_id: str, fields: str = FIELDS) -> dict:
    """一次请求拿全。返回原样响应，外加一层 _fetch 溯源。"""
    t0 = time.monotonic()
    status, payload = s2_call(f"/paper/{paper_id}", {"fields": fields})
    if status != 200:
        raise RuntimeError(f"/paper/{paper_id} -> {status}: {payload}")
    refs, cites = payload.get("references") or [], payload.get("citations") or []
    payload["_fetch"] = {
        "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "api_base": S2_BASE, "key_origin": KEY_ORIGIN, "fields": fields,
        "elapsed_s": round(time.monotonic() - t0, 2),
        # 触顶就说明这篇的引用关系不完整，要改走 /paper/{id}/citations 分页
        "truncated": {"references": len(refs) >= NESTED_CAP, "citations": len(cites) >= NESTED_CAP},
        "note": "计数为 fetched_at 时刻快照；venue/year 可能来自预印本，非发表信息；"
                "引用边属性（contexts/intents/isInfluential）不在本响应内。",
    }
    return payload

## 跑清单

每篇写成 `_scratch/s2/<key>.json`（原样响应 + `_fetch` 溯源），另出一份 `index.json`
记录每篇的解析方式与回执 —— 解析入口是唯一依赖 S2 匹配行为的一步，它的结论要留痕。

已有文件**跳过**而不是覆盖，所以这个 cell 可以中断后重跑；要刷新某篇就删掉它的 json。

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

index = []
for r in resolved:
    out_path = OUT_DIR / f"{r['key']}.json"
    if r["paper_id"] is None:
        print(f"  skip  {r['key']:28} 未解析：{r['error']}")
        index.append(r | {"file": None}); continue
    if out_path.exists():                                   # 断点续跑
        paper = json.loads(out_path.read_text(encoding="utf-8"))
        print(f"  have  {r['key']:28} 已有产物，跳过")
    else:
        paper = fetch_paper(r["paper_id"])
        out_path.write_text(json.dumps(paper, ensure_ascii=False, indent=1), encoding="utf-8")
        f = paper["_fetch"]
        print(f"  ok    {r['key']:28} {out_path.stat().st_size / 1024:>7.0f} KiB "
              f"{f['elapsed_s']:>5.1f}s  ref {len(paper['references']):>4} / "
              f"cite {len(paper['citations']):>4}"
              + ("  ⚠ 触顶 1000，引用关系不完整" if any(f["truncated"].values()) else ""))
    index.append(r | {"file": out_path.name,
                      "title": paper["title"],
                      "fetched_at": paper["_fetch"]["fetched_at"],
                      "counts": {"references": len(paper["references"]),
                                 "citations": len(paper["citations"]),
                                 "citationCount": paper["citationCount"]},
                      "truncated": paper["_fetch"]["truncated"]})

(OUT_DIR / "index.json").write_text(json.dumps(index, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n{sum(1 for i in index if i['file'])}/{len(index)} 篇 → {OUT_DIR.relative_to(REPO)}")
print("回填用的 paperId：")
for i in index:
    if i["paper_id"]:
        print(f'  {{"key": "{i["key"]}", "s2": "{i["paper_id"]}"}},')